**Данные, с которыми работает пайплайн**

### Вход
- `baseline_blocks.pickle` — кварталы до проекта.
- `scenario_blocks.pickle` — сценарные кварталы после реализации.
- `context_blocks.pickle` — контекстные кварталы города для учета соседних характеристик.



In [ ]:
import pandas as pd
baseline_blocks = pd.read_pickle('./data/baseline_blocks.pickle')
scenario_blocks = pd.read_pickle('./data/scenario_blocks.pickle')
context_blocks = pd.read_pickle('./data/context_blocks.pickle')

In [ ]:
baseline_blocks.head()

In [ ]:
scenario_blocks.head()

In [ ]:
context_blocks.head()

In [ ]:
m = baseline_blocks.explore(
    column='land_use',
    legend=True,
    style_kwds={
        "weight": 0.5,     # толщина границ
        "color": "black",  # цвет границ
        "fillOpacity": 0.7
    },
)

m

In [ ]:
m = scenario_blocks.explore(
    column='land_use',
    legend=True,
    style_kwds={
        "weight": 0.5,     # толщина границ
        "color": "black",  # цвет границ
        "fillOpacity": 0.7
    },
)

m

## Данные по терртории до проекта

In [ ]:
from urbanomy.methods.land_value_modeling.land_data_preparation import LandDataPreparator

preparator = LandDataPreparator(
    scenario_blocks_source=baseline_blocks,
    context_blocks_source=context_blocks,
)

basline_prepared_blocks = preparator.prepare()
basline_prepared_blocks.head()

## Данные по терртории после реализации проекта

In [ ]:
preparator = LandDataPreparator(
    scenario_blocks_source=scenario_blocks,
    context_blocks_source=context_blocks,
)

scenario_prepared_blocks = preparator.prepare()
scenario_prepared_blocks.head()

## Сохранение данных

In [ ]:
basline_prepared_blocks.to_pickle('./data/baseline_prepared_blocks.pickle')
scenario_prepared_blocks.to_pickle('./data/scenario_prepared_blocks.pickle')


# Предсказание стоимости земли

## Загружаем данные

In [ ]:
baseline_blocks = pd.read_pickle('./data/baseline_prepared_blocks.pickle')
scenario_blocks = pd.read_pickle('./data/scenario_prepared_blocks.pickle')

## Предсказываем стоимость земли до проекта

In [ ]:
from catboost import CatBoostRegressor

from urbanomy.methods.land_value_modeling import LandPriceEstimator
model = CatBoostRegressor()

model.load_model('./data/catboost_model.cbm')


estimator = LandPriceEstimator(
    model=model,
    blocks=baseline_blocks,
)
baseline_blocks = estimator.predict()
baseline_blocks.head()


## Предсказываем стоимость земли после реализации проекта

In [ ]:
from urbanomy.methods.land_value_modeling import LandPriceEstimator

estimator = LandPriceEstimator(
    model=model,
    blocks=scenario_blocks, #or baseline_blocks
)
scenario_blocks = estimator.predict()
scenario_blocks.head()

## Получаем статистику

In [ ]:
import geopandas as gpd
import pandas as pd

from urbanomy.methods.land_value_modeling import transfer_baseline_prices



blocks_full_value = transfer_baseline_prices(scenario_blocks, baseline_blocks)

blocks_full_value.to_pickle('./data/blocks_full_value.pickle')

# фильтруем только сценарные кварталы
scn_blocks = blocks_full_value.loc[blocks_full_value['is_project'] == True].copy()


sum_before = scn_blocks['land_value_before'].sum()
sum_after  = scn_blocks['land_value'].sum()

print(f"Сценарные кварталы — суммарная стоимость ДО застройки: {sum_before:,.0f} руб.")
print(f"Сценарные кварталы — суммарная стоимость ПОСЛЕ застройки: {sum_after:,.0f} руб.")

In [ ]:
from urbanomy.methods.land_value_modeling import plot_land_price_maps

visualization_output = plot_land_price_maps(
    blocks_pred=blocks_full_value,
    price_column="land_value",  # используйте "land_value_before" для цен до застройки
    buffer_radius_m=2000,
)

In [ ]:
m = scn_blocks.explore(
    column='land_value',
    cmap="coolwarm",
    legend=True,
    style_kwds={
        "weight": 0.5,     # толщина границ
        "color": "black",  # цвет границ
        "fillOpacity": 0.7
    },
    name="INV"
)

m

## Сценарий изменения ТЭП участка

In [ ]:
from urbanomy.methods.land_value_modeling import (
    ScenarioTEPModifier,
    plot_scenario_impact,
)

blocks_before = scenario_blocks
changes = {
    'land_use': 'LandUse.RESIDENTIAL',
    'share': 0.95,
    'footprint_area': 49039.24,
    'build_floor_area': 230279.19,
    'living_area': 155896.85,
    'population': 5168
}
target_idx = 62 #63

modifier = ScenarioTEPModifier(blocks_before)
blocks_after = modifier.apply(target_idx, changes)

scenario_result = plot_scenario_impact(
    blocks_before=blocks_before,
    blocks_after=blocks_after,
    model=model,
    target_idx=target_idx,
    figsize=(25, 25),
)

# Оценка инвестиционной привлекательности

In [ ]:
# import requests
# import pandas as pd
# from typing import Final
# from blocksnet.enums import LandUse


# URBAN_API = "https://urban-api.idu.kanootoko.org/api/v1"

# #211
# #198
# #124
# #335

# id = 198

# scenario_indicators_response = requests.get(
#     f"{URBAN_API}/scenarios/{id}/indicators_values", verify=False
# )
# if scenario_indicators_response.status_code != 200:
#     raise Exception("Ошибка при получении индикаторов по сценарию")

# scenario_indicators = scenario_indicators_response.json()

# indicator_attributes = {
#     indicator['indicator']['name_full']: indicator['value']
#     for indicator in scenario_indicators
# }


# LAND_USE_RULES: Final[dict[str, LandUse]] = {
#     'Потенциал развития среднеэтажной жилой застройки': LandUse.RESIDENTIAL,
#     "Потенциал развития застройки общественно-деловой зоны": LandUse.BUSINESS,
#     "Потенциал развития застройки рекреационной зоны": LandUse.RECREATION,
#     "Потенциал развития застройки зоны специального назначения": LandUse.SPECIAL,
#     "Потенциал развития застройки промышленной зоны": LandUse.INDUSTRIAL,
#     "Потенциал развития застройки сельскохозяйственной зоны": LandUse.AGRICULTURE,
#     "Потенциал развития застройки транспортной зоны": LandUse.TRANSPORT,
# }

# records: list[dict[str, object]] = []
# for indicator_name, land_use in LAND_USE_RULES.items():
#     potential = indicator_attributes.get(indicator_name)
#     if potential is None:
#         continue
#     records.append({
#         'land_use': land_use,
#         'potential': potential,
#     })

# potential_df = pd.DataFrame(records).reset_index(drop=True)
# potential_df.to_csv("./data/land_use_potentials.csv", index=False)

# potential_df

In [ ]:
from urbanomy.methods.investment_potential import prepare_investment_input
# potential_df = pd.read_csv("./data/land_use_potentials.csv")

scenario_blocks = pd.read_pickle('./data/blocks_full_value.pickle').to_crs(32636)
scenario_blocks = scenario_blocks.loc[scenario_blocks['is_project'] == True].copy()

potential_df = pd.read_csv("./data/land_use_potentials.csv")

In [ ]:
investment_input = prepare_investment_input(
    gdf = scenario_blocks,
)

investment_input.head()

In [ ]:
from blocksnet.enums import LandUse

benchmarks_demo = {
    LandUse.RESIDENTIAL: {
        "cost_build": 45_000,
        "price_sale": 120_000,
        "construction_years": 3,
        "sale_years": 3,
        "opex_rate": 800,
    },
    LandUse.BUSINESS: {
        "cost_build": 55_000,
        "rent_annual": 25_000,
        "rent_years": 12,
        "construction_years": 4,
        "opex_rate": 1_300,
    },
    LandUse.RECREATION: {
        "cost_build": 20_000,
        "rent_annual": 4_500,
        "rent_years": 15,
        "construction_years": 3,
        "opex_rate": 1_000,
    },
    LandUse.SPECIAL: {
        "cost_build": 35_000,
        "rent_annual": 11_000,
        "rent_years": 15,
        "construction_years": 3,
        "opex_rate": 1_500,
    },
    LandUse.INDUSTRIAL: {
        "cost_build": 38_000,
        "rent_annual": 14_800,
        "rent_years": 12,
        "construction_years": 3,
        "opex_rate": 700,
    },
    LandUse.AGRICULTURE: {
        "cost_build": 25_000,
        "rent_annual": 6_500,
        "rent_years": 15,
        "construction_years": 3,
        "opex_rate": 300,
    },
    LandUse.TRANSPORT: {
        "cost_build": 18_000,
        "rent_annual": 6_200,
        "rent_years": 15,
        "construction_years": 3,
        "opex_rate": 600,
    },
}


## Вычисляем оценку

In [ ]:
from urbanomy.methods.investment_potential import InvestmentAttractivenessAnalyzer

an = InvestmentAttractivenessAnalyzer(benchmarks=benchmarks_demo)
summary = an.calculate_investment_metrics(investment_input, discount_rate=0.18)
summary


In [ ]:
scn = scenario_blocks[['geometry']].join(summary)
scn.to_file('./data/blocks_investment.geojson', driver='GeoJSON')

## Визуализация

In [ ]:
column = "INV"


# ---------- Статичная картинка с тонкими границами ----------
ax = scn.plot(
    column=column,
    cmap="RdYlGn",
    legend=True,
    figsize=(15, 10),
    edgecolor="black",   # цвет границ
    linewidth=0.3        # толщина границ
)

for _, row in scn.dropna(subset=[column]).iterrows():
    x, y = row.geometry.representative_point().coords[0]
    ax.text(
        x,
        y,
        f"{row.land_use}\n{row[column]:.2f}",
        ha="center",
        va="center",
        fontsize=6,
        color="black",
    )

ax.set_axis_off()


In [ ]:
m = scn.explore(
    column='INV',
    cmap="RdYlGn",
    legend=True,
    style_kwds={
        "weight": 0.5,     # толщина границ
        "color": "black",  # цвет границ
        "fillOpacity": 0.7
    },
    name="INV"
)

m

## Влияние проекта на показатели социально-экономического развития муниципального образования

In [ ]:
gdf = gpd.read_file('./data/blocks_investment.geojson')
gdf.head()

In [ ]:
from urbanomy.methods.socio_economic_indicators.constants import (
    DEFAULT_VA_PER_M2_OPS,
    DEFAULT_JOBS_PER_M2,
    DEFAULT_WAGE_BY_USE,
    DEFAULT_PROFIT_SHARE_OPS,
    DEFAULT_CAPEX_CAPITALIZABLE_SHARE,
    DEFAULT_AMORTIZATION_RATES,
)

project_cfg = {  # Эффекты стройки считаем разово по всему проекту

    "population": 520_000,  # Население базовой территорииА
    "employment_share": 0.62,  # Доля занятых в базе (по умолчанию 0.62)
    "avg_wage_base": 74_000,  # Средняя зарплата до проекта
    "build_wage_share": 0.27,  # Доля зарплаты в инвестициях на стройке
    "build_profit_margin": 0.065,  # Рентабельность строительства
    "tax_rates": {  # Налоговые ставки проекта
        "pit": 0.12,  # НДФЛ
        "cit": 0.17,  # налог на прибыль
        "prop": 0.025,  # налог на имущество
        "land": 0.015,  # земельный налог
    },
    "va_coeff_build": {  # Коэффициенты валовой добавленной стоимости на стройке
        "default": 0.50,  # базовый коэффициент для прочих категорий
        "business": 0.55,  # повышаем мультипликатор для деловой застройки
        "special": 0.52,
    },
    "va_per_m2_ops": {  # ВДС на квадратный метр в эксплуатации
        **DEFAULT_VA_PER_M2_OPS,
        "business": 13_500.0,
        "transport": 5_500.0,
    },
    "jobs_per_m2": {  # Рабочие места на квадратный метр
        **DEFAULT_JOBS_PER_M2,
        "business": 1 / 20,
        "special": 1 / 28,
    },
    "wage_by_use": {  # Средние зарплаты по видам использования земли
        **DEFAULT_WAGE_BY_USE,
        "business": 92_000,
        "transport": 63_000,
    },
    "profit_share_ops": {  # Доли прибыли в обороте в эксплуатации
        **DEFAULT_PROFIT_SHARE_OPS,
        "business": 0.20,
        "special": 0.16,
    },
    "capex_capitalizable_share": {  # Доля капзатрат, переходящих в основные средства
        **DEFAULT_CAPEX_CAPITALIZABLE_SHARE,
        "transport": 0.92,
    },
    "amortization_rates": {  # Годовые нормы амортизации
        **DEFAULT_AMORTIZATION_RATES,
        "business": 0.032,
        "special": 0.045,
}}


In [ ]:
from urbanomy.methods.socio_economic_indicators.sei_calculate import SEREstimator

deafaut_cfg = {
    "population": 300_000,
}

est = SEREstimator(deafaut_cfg) # or use project_cfg
result = est.compute(gdf, pretty=True)
result